# AI-Lake Format — Interactive Demo

Comprehensive walkthrough of the AI-Lake Python SDK:

| Section | What you will learn |
|---|---|
| 1 | **Write** — `open_table()` fluent API, insert, commit |
| 2 | **Load fixture** — pre-generated demo data (500 rows, dim=32) |
| 3 | **Search** — `SearchQuery`, pointer-only results |
| 4 | **Full row data** — `fetch_data=True`, Arrow / Pandas |
| 5 | **Fluent API** — `limit()`, `to_arrow()`, `to_list()`, `to_polars()` |
| 6 | **HNSW tuning** — `pre_normalize`, `normalized_cosine`, `hnsw_m` |
| 7 | **Idempotent writes** — `write_batch_idempotent` |
| 8 | **Iceberg compat** — PyArrow + PyIceberg (no plugin) |
| 9 | **SQL** — DuckDB direct Parquet scan |
| 10 | **RAG context** — `assemble_context()` for LLMs |
| 11 | **Object storage** — MinIO / S3 via PyArrow |
| 12 | **IVF-PQ index** — `pq_only=True`, product quantization |
| 13 | **Residual PQ** — `ivf_residual=True`, better recall |
| 14 | **Deferred write** — `write_batch_auto_deferred`, ~200k vec/s |
| 15 | **HNSW tuning** — M, ef_construction, normalized_cosine |
| 16 | **Async API** — concurrent multi-table search |
| 17 | **Storage estimator** — capacity planning math |
| 17B | **Vector precision** — `precision=` F16 / F32 / I8, size + recall trade-off |
| 18 | **Embedding model tracking** — `embedding_model`, Iceberg properties |
| 19 | **Pattern B** — `embed_fn`: auto-embed |
| 20 | **Migration** — `migrate_embeddings()` |
| 21 | **N vector columns** — `VectorColSpec`, `write_batch_multi` |
| 22 | **Cross-modal search** — `search_multimodal`, RRF fusion |
| 23 | **`MultimodalContextSchema`** — column name constants |
| 24 | **Agent memory** — `ailake.Agent`, `remember`, `recall`, `log_tool_call` |
| 25 | **Partition isolation** — `partition_by`, `partition_filter` |
| 26 | **Hybrid scoring** — `ScoreFn`, distance × recency × importance |
| 27 | **`ToolCallSchema`** — tool call logging and semantic search |
| 28 | **`EpisodicMemorySchema`** — recency decay, `recency_weight` |
| 29 | **`delete_where`** — Iceberg equality delete |
| 30 | **Schema evolution** — `add_column`, `rename_column`, `evolve_schema` |
| 30B | **Vector column backfill** — `add_vector_column` + `backfill_vector_column` |
| 31 | **Compaction** — `ailake.compact()`; partition fields + `format_version=3` |
| 32 | **Tantivy FTS intro** — `fts_text_columns`, `search_text()`, O(log N) fast path (→ `11_fts.ipynb` for full demo) |


## 0. Setup

In [ ]:
import io
import json
import os
import pathlib

import ailake
import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

TABLE_PATH  = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')
QUERY_PATH  = pathlib.Path(TABLE_PATH).parent / 'demo_query.json'
CUSTOM_PATH = str(pathlib.Path(TABLE_PATH).parent / 'ailake_custom')

print(f'Demo table  : {TABLE_PATH}')
print(f'Custom table: {CUSTOM_PATH}')

## 1. Write — `open_table()` fluent API

`ailake.open_table()` opens or creates an AI-Lake table. The `Table` handle:
- `insert(texts, embeddings)` — buffer a batch (`list[str]` + numpy array or list)
- `commit()` — persist as a new Iceberg snapshot, returns snapshot ID

In Jupyter, `table` renders as an HTML card showing path and vector config.

In [ ]:
# Generate 10 synthetic unit vectors (dim=32 — same as demo fixture)
DIM = 32
rng = np.random.default_rng(seed=42)

topics = [
    "transformer architecture", "retrieval-augmented generation",
    "vector databases", "semantic search", "LLM fine-tuning",
    "approximate nearest neighbor", "columnar storage", "data versioning",
    "stream processing", "knowledge graphs",
]
texts = [f'Custom doc about {t}.' for t in topics]
embs  = rng.standard_normal((len(texts), DIM)).astype(np.float32)
embs /= np.linalg.norm(embs, axis=1, keepdims=True)   # unit vectors

table = ailake.open_table(CUSTOM_PATH, dim=DIM, metric='cosine')
table.insert(texts, embs)
snap_id = table.commit()

print(f'Snapshot ID : {snap_id}')
print(f'Rows written: {len(texts)}')
table   # HTML card in Jupyter

In [ ]:
# Context manager form — equivalent, auto-cleanup
with ailake.open_table(CUSTOM_PATH, dim=DIM, metric='cosine') as t:
    t.insert(['Extra row about caching.'], [embs[0].tolist()])
    snap2 = t.commit()
print(f'Second snapshot: {snap2}')

## 1B. Create an empty table — `ailake.create_table()`

Creates a schema-only Iceberg table (no data files) — useful for provisioning a table
ahead of a separate ingestion job, or for frameworks that need a table to exist before
the first write. Immediately usable by `open_table()` / `TableWriter` afterward.

Also reachable from DuckDB (`ailake_create_table()` SQL function) and the CLI
(`ailake create`); the Spark/Trino/Flink native bridge exists (`AilakeNative.createTable`)
but isn't wired into SQL `CREATE TABLE` DDL yet — see `docs/specs/JVM_PLUGINS.md`.

In [ ]:
EMPTY_PATH = str(pathlib.Path(TABLE_PATH).parent / 'ailake_empty')

ok = ailake.create_table(EMPTY_PATH, dim=DIM, metric='cosine')
print(f'create_table ok: {ok}')

# Fails loudly on a duplicate — no silent no-op
try:
    ailake.create_table(EMPTY_PATH, dim=DIM, metric='cosine')
except Exception as e:
    print(f'Second call raises (table exists): {e}')

# The table is immediately usable by the normal write path
table = ailake.open_table(EMPTY_PATH, dim=DIM, metric='cosine')
table.insert(['first row in a table created empty'], [rng.standard_normal(DIM).astype(np.float32).tolist()])
snap = table.commit()
print(f'Snapshot after first write: {snap}')

## 2. Load pre-generated demo fixture

`init_demo.py` wrote 500 rows (dim=32) to `TABLE_PATH` at container start.
It saved the embedding of doc_id=0 to `demo_query.json` so we can verify that
the nearest neighbour of a vector is itself (expected distance ≈ 0).

In [ ]:
with open(QUERY_PATH) as f:
    demo = json.load(f)

DIM_FIXTURE = int(demo['dim'])
query_vec   = np.array(demo['query_vector'], dtype=np.float32)
print(f'Query dim      : {DIM_FIXTURE}')
print(f'Expected top-1 : "{demo["expected_top1_text"][:70]}..."')

## 3. Pointer-only search — `SearchQuery`

`ailake.search()` returns a **`SearchQuery`** — lazy, chainable, no I/O until materialised.

Pointer-only mode (default): `row_id`, `distance`, `file` columns only.
Only the HNSW index is touched; no Parquet column data is loaded.

In [ ]:
results = ailake.search(TABLE_PATH, query_vec, top_k=5)
print(results)          # SearchQuery(5 results, top_k=5)

df_ptr = results.to_pandas()
df_ptr

In [ ]:
# Verify: query vector is its own nearest neighbour (distance ≈ 0)
top = ailake.search(TABLE_PATH, query_vec, top_k=1).to_list()[0]
assert top['distance'] < 0.01, f'Expected near-zero, got {top["distance"]}'
print(f'PASS — distance = {top["distance"]:.2e}  (query found itself)')

## 4. Full row data — `fetch_data=True`

`fetch_data=True` loads all Parquet columns for the matched rows.
Returns a `pyarrow.Table` with all original columns **plus** `_distance` (Float32).

Use when you need the actual text / metadata without a full table scan.

In [ ]:
df_full = ailake.search(TABLE_PATH, query_vec, top_k=5, fetch_data=True).to_pandas()
# Shows all Parquet columns (text, embedding, …) + _distance
df_full[['text', '_distance']]

In [ ]:
arrow_tbl = ailake.search(TABLE_PATH, query_vec, top_k=5, fetch_data=True).to_arrow()
print(f'Schema : {arrow_tbl.schema}')
print(f'Rows   : {arrow_tbl.num_rows}')
arrow_tbl.to_pandas()[['text', '_distance']]

## 5. Fluent API patterns

`SearchQuery` chains: `limit()` → `to_pandas()` / `to_arrow()` / `to_polars()` / `to_list()`.
All materialise lazily on first call; subsequent calls reuse the cached result.

In [ ]:
# .limit() caps k before materialising — still no I/O beyond the index
top3 = ailake.search(TABLE_PATH, query_vec, top_k=10).limit(3).to_pandas()
top3

In [ ]:
# to_list() always returns pointer-only dicts regardless of fetch_data
items = ailake.search(TABLE_PATH, query_vec, top_k=3).to_list()
for i, r in enumerate(items):
    fname = pathlib.Path(r['file']).name
    print(f'  #{i+1}  row_id={r["row_id"]:>4}  distance={r["distance"]:.6f}  file={fname}')

In [ ]:
# to_arrow() pointer-only: pyarrow.Table with row_id, distance, file columns
arrow_ptr = ailake.search(TABLE_PATH, query_vec, top_k=5).to_arrow()
print(f'Columns: {arrow_ptr.schema.names}')
arrow_ptr.to_pandas()

In [ ]:
# to_polars() — polars.DataFrame (pointer-only)
try:
    import polars as pl
    lf = ailake.search(TABLE_PATH, query_vec, top_k=5).to_polars()
    print(f'polars shape: {lf.shape}')
    lf
except ImportError:
    print('polars not installed — pip install polars')

In [ ]:
# Table handle search — search via an already-opened Table object
demo_table = ailake.open_table(TABLE_PATH, dim=DIM_FIXTURE, metric=demo['metric'])
df_handle  = demo_table.search(query_vec, top_k=3, fetch_data=True).to_pandas()
df_handle[['text', '_distance']]

## 6. HNSW tuning — `pre_normalize` + `normalized_cosine`

`pre_normalize=True` normalises vectors to unit-L2 at **write** time and uses
`1 - dot(a, b)` in the HNSW hot loop instead of full cosine (~12-20 % faster
for dim ≥ 1024). Query vectors are normalised automatically at search time.

`metric="normalized_cosine"` disables F16 quantisation (exact F32 stored)
and activates the dot-product hot path.

HNSW graph parameters:

| `hnsw_m` | `hnsw_ef_construction` | Preset |
|---|---|---|
| 8 | 100 | Low latency / high QPS |
| 16 | 150 | General purpose (default) |
| 24 | 200 | High recall (RAG) |
| 32 | 400 | Max recall (medical/legal) |

In [ ]:
NORM_PATH = str(pathlib.Path(TABLE_PATH).parent / 'ailake_normalized')

table_norm = ailake.open_table(
    NORM_PATH,
    dim=DIM,
    metric='normalized_cosine',  # fast 1-dot(a,b); F16 quant disabled
    pre_normalize=True,           # normalise at write time
    hnsw_m=24,
    hnsw_ef_construction=200,
)
table_norm.insert(texts, embs)
table_norm.commit()
print('Normalized table written.')
table_norm

In [ ]:
# Search on normalized table — query is auto-normalised at search time
results_norm = ailake.search(NORM_PATH, embs[0], top_k=3, fetch_data=True).to_pandas()
results_norm[['text', '_distance']]

## 7. Idempotent writes — `write_batch_idempotent`

`TableWriter.write_batch_idempotent(texts, embs, batch_id)` is a no-op if
`batch_id` was already committed. Safe to retry — useful for Airflow re-runs,
at-least-once delivery pipelines, and streaming connectors.

In [ ]:
import shutil
shutil.rmtree(str(pathlib.Path(TABLE_PATH).parent / "ailake_idempotent"), ignore_errors=True)

from ailake import TableWriter

IDEM_PATH   = str(pathlib.Path(TABLE_PATH).parent / 'ailake_idempotent')
extra_texts = ['Idempotent doc A about caching.', 'Idempotent doc B about retries.']
extra_embs  = rng.standard_normal((2, DIM)).astype(np.float32)
extra_embs /= np.linalg.norm(extra_embs, axis=1, keepdims=True)

def _write(batch_id: str) -> int:
    w = TableWriter(IDEM_PATH, dim=DIM, metric='cosine')
    w.write_batch_idempotent(extra_texts, extra_embs.tolist(), batch_id)
    return w.commit()

snap_a = _write('pipeline-run-2026-001')
print(f'First  write : snapshot_id={snap_a}')

snap_b = _write('pipeline-run-2026-001')   # same id — write skipped
print(f'Retry  write : snapshot_id={snap_b}  (batch_id already committed)')

snap_c = _write('pipeline-run-2026-002')   # new id — committed
print(f'New    write : snapshot_id={snap_c}  (new batch_id)')

## 8. Iceberg Compatibility — PyArrow + PyIceberg

The same `.parquet` files are valid Apache Iceberg Spec v2.
Standard readers see all tabular columns; the HNSW footer section is silently ignored.

In [ ]:
# PyArrow — direct Parquet read, zero configuration
parquet_files = sorted(pathlib.Path(TABLE_PATH).rglob('*.parquet'))
parts         = [pq.read_table(str(p)) for p in parquet_files]
arrow_table   = pa.concat_tables(parts)

print(f'Files  : {len(parquet_files)}')
print(f'Rows   : {len(arrow_table)}')
print(f'Schema : {arrow_table.schema}')
arrow_table.to_pandas().head(3)

In [ ]:
# PyIceberg — SqlCatalog backed by SQLite (no Hive/REST required)
from pyiceberg.catalog.sql import SqlCatalog

catalog = SqlCatalog('demo', **{
    'uri'       : 'sqlite:////tmp/demo_iceberg.db',
    'warehouse' : f'file://{TABLE_PATH}',
})

try:
    catalog.create_namespace('default')
except Exception:
    pass

meta_dir = pathlib.Path(TABLE_PATH) / 'default' / 'table' / 'metadata'
hint     = (meta_dir / 'version-hint.text').read_text().strip()
meta_loc = f'file://{meta_dir}/v{hint}.metadata.json'

try:
    catalog.drop_table('default.table')
except Exception:
    pass
catalog.register_table('default.table', meta_loc)

iceberg_tbl = catalog.load_table('default.table')
print(f'Schema: {iceberg_tbl.schema()}')
df_iceberg  = iceberg_tbl.scan().to_pandas()
print(f'Rows  : {len(df_iceberg)}')
df_iceberg.head(3)

## 9. SQL Compatibility — DuckDB

DuckDB reads AI-Lake files as standard Parquet. The `embedding` column appears
as `BLOB` (raw F16 bytes) — the HNSW section in the footer is invisible.

In [ ]:
parquet_glob = str(pathlib.Path(TABLE_PATH) / '**' / '*.parquet')
con = duckdb.connect()

con.execute(
    f"SELECT count(*) AS total_rows, avg(length(text)) AS avg_text_len,"
    f" avg(octet_length(embedding)) AS avg_emb_bytes"
    f" FROM read_parquet('{parquet_glob}', hive_partitioning=false)"
).df()

In [ ]:
# Full-text filter — pushed down to Parquet column scan
con.execute(
    f"SELECT text FROM read_parquet('{parquet_glob}', hive_partitioning=false)"
    f" WHERE text LIKE '%vector search%' LIMIT 5"
).df()

In [ ]:
# Schema inspection — DuckDB shows embedding as BLOB
con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{parquet_glob}', hive_partitioning=false) LIMIT 1"
).df()

## 10. RAG Context Assembly — `assemble_context()`

`ailake.assemble_context()` converts search results into structured XML context
for LLM input. It deduplicates near-identical chunks and respects a token budget.
Returns `{"text": str, "chunk_count": int, "token_estimate": int}` (Fase 15) —
not a plain string.


In [ ]:
full_results = ailake.search(TABLE_PATH, query_vec, top_k=10, fetch_data=True).to_pandas()

chunks = [
    {
        'document_id'  : str(i),
        'chunk_index'  : 0,
        'chunk_text'   : row['text'],
        'document_title': f'Document {i}',
        'distance'     : float(row['_distance']),
    }
    for i, (_, row) in enumerate(full_results.iterrows())
]

context_result = ailake.assemble_context(
    chunks=chunks,
    max_tokens=2048,
    dedup_threshold=0.1,
)
print(f"chunk_count={context_result['chunk_count']}  token_estimate={context_result['token_estimate']}")
print(context_result['text'][:1200])

In [ ]:
# Richer chunk metadata — title, section path, source URI
chunks_rich = [
    {
        'document_id'   : 'doc-001',
        'chunk_index'   : 0,
        'chunk_text'    : 'AI-Lake stores vectors and tabular data in a single Parquet file.',
        'document_title': 'AI-Lake Overview',
        'section_path'  : 'Introduction > Format',
        'source_uri'    : 's3://my-lake/docs/overview.pdf',
        'distance'      : 0.05,
    },
    {
        'document_id'   : 'doc-001',
        'chunk_index'   : 1,
        'chunk_text'    : 'The HNSW index lives in the Parquet footer, invisible to standard readers.',
        'document_title': 'AI-Lake Overview',
        'section_path'  : 'Introduction > Compatibility',
        'source_uri'    : 's3://my-lake/docs/overview.pdf',
        'distance'      : 0.12,
    },
]
print(ailake.assemble_context(chunks=chunks_rich, max_tokens=1024, dedup_threshold=0.05)['text'])

## 11. Object Storage — MinIO / S3

Upload the local AI-Lake Parquet files to MinIO (local S3 emulator) and read
them back with PyArrow S3 filesystem — demonstrating object-storage compatibility.

> MinIO console: http://localhost:9001  (user/pass: minioadmin)

In [ ]:
import boto3
from botocore.client import Config

MINIO_ENDPOINT = os.environ.get('MINIO_ENDPOINT', 'http://minio:9000')
ACCESS_KEY     = os.environ.get('MINIO_ACCESS_KEY', 'minioadmin')
SECRET_KEY     = os.environ.get('MINIO_SECRET_KEY', 'minioadmin')
BUCKET         = 'demo-bucket'

s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    config=Config(signature_version='s3v4'),
)

uploaded = []
for local_path in parquet_files:
    key = 'ailake_demo/' + str(local_path.relative_to(TABLE_PATH))
    s3.upload_file(str(local_path), BUCKET, key)
    uploaded.append(key)
    print(f'  -> s3://{BUCKET}/{key}')

print(f'\nTotal: {len(uploaded)} files uploaded')

In [ ]:
import pyarrow.fs as pafs

s3fs = pafs.S3FileSystem(
    endpoint_override=MINIO_ENDPOINT.replace('http://', ''),
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    scheme='http',
)

s3_tables   = [pq.read_table(f'{BUCKET}/{k}', filesystem=s3fs) for k in uploaded]
s3_combined = pa.concat_tables(s3_tables)

print(f'Rows read from MinIO: {len(s3_combined)}')
s3_combined.to_pandas().head(3)

## 12. IVF-PQ — Product Quantization index

PQ compresses each vector into a small fixed-size code using sub-vector codebooks.

| Mode | Storage per vector (dim=32) | Raw vectors kept? | Reranking? |
|---|---|---|---|
| HNSW (default) | 64 bytes F16 | Yes | Yes (exact) |
| PQ + raw | 64 bytes F16 + 4 bytes code | Yes | Yes |
| `pq_only=True` | 4 bytes code only | **No** | No |

At dim=1536 the savings are dramatic: 3 072 bytes → 48 bytes/vec (98.4 % less).


In [ ]:
import pathlib, os
import ailake
import numpy as np

BASE   = pathlib.Path(os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')).parent
DIM    = 32
rng    = np.random.default_rng(42)
texts  = [f'PQ demo doc {i}.' for i in range(50)]
embs   = rng.standard_normal((50, DIM)).astype(np.float32)
embs  /= np.linalg.norm(embs, axis=1, keepdims=True)

# Standard write — raw vectors kept (default)
hnsw_path = str(BASE / 'nb_hnsw')
ailake.open_table(hnsw_path, dim=DIM, metric='cosine').insert(texts, embs).commit()

# PQ-only — raw vectors discarded after index build
pq_path = str(BASE / 'nb_pq_only')
ailake.open_table(pq_path, dim=DIM, metric='cosine', pq_only=True).insert(texts, embs).commit()

def dir_bytes(p):
    return sum(f.stat().st_size for f in pathlib.Path(p).rglob('*') if f.is_file())

hnsw_kb = dir_bytes(hnsw_path) / 1024
pq_kb   = dir_bytes(pq_path)   / 1024
print(f'HNSW table : {hnsw_kb:.1f} KB  (50 rows, raw F16 + HNSW graph)')
print(f'PQ-only    : {pq_kb:.1f} KB  (50 rows, PQ codes only)')
print(f'Ratio      : {hnsw_kb / pq_kb:.1f}× smaller with pq_only')


In [ ]:
# Search works the same way on both tables
q = embs[0].tolist()

r_hnsw = ailake.search(hnsw_path, q, top_k=3).to_pandas()
r_pq   = ailake.search(pq_path,   q, top_k=3).to_pandas()

print('HNSW results:')
print(r_hnsw[['row_id', 'distance']])
print()
print('PQ-only results:')
print(r_pq[['row_id', 'distance']])
print()
print('(PQ distances slightly less precise — trade-off for storage savings)')


### `rerank_factor` — exact-distance reranking after IVF-PQ

IVF-PQ codes are lossy (8-bit sub-quantizer codes). `search(..., rerank_factor=N)`
fetches `top_k * N` candidates by approximate PQ distance, then reranks them with
exact F32 distances computed from the raw vectors kept alongside the index (not
available on a `pq_only=True` table — see the storage/reranking table above).


In [ ]:
# Build an IVF-PQ table (raw vectors kept -> exact reranking available)
ivfpq_path = str(BASE / 'nb_ivf_pq')
w = ailake.TableWriter(ivfpq_path, dim=DIM, metric='cosine')
w.write_batch_ivf_pq(texts, embs.tolist())
w.commit()

query = embs[0].tolist()

# Ground truth: brute-force exact cosine distance
exact_dist = 1.0 - embs @ np.array(query)
exact_top3 = np.argsort(exact_dist)[:3]

# Pointer-only search (fetch_data=False) carries row_id; fetch_data=True returns
# Parquet columns + _distance instead (no row_id — see section 4).
r_no_rerank = ailake.search(ivfpq_path, query, top_k=3)
r_reranked  = ailake.search(ivfpq_path, query, top_k=3, rerank_factor=5)

print(f'Exact top-3 row_ids (brute force): {list(exact_top3)}')
print()
print('IVF-PQ search, no rerank_factor (approximate PQ distances):')
for r in r_no_rerank.to_list():
    print(f"  row_id={r['row_id']:3d}  distance={r['distance']:.4f}")
print()
print('IVF-PQ search, rerank_factor=5 (exact F32 distances on top-15 candidates):')
for r in r_reranked.to_list():
    print(f"  row_id={r['row_id']:3d}  distance={r['distance']:.4f}")


## 13. Residual PQ — per-cluster encoding

`ivf_residual=True` encodes `vec - cluster_centroid` instead of the raw vector.
Residuals cluster tightly → PQ codebook fits them better → ~2-4 pp better recall@10
at the same code size.  Useful for high-accuracy RAG pipelines.


In [ ]:
res_path = str(BASE / 'nb_residual_pq')
ailake.open_table(res_path, dim=DIM, metric='cosine', ivf_residual=True)       .insert(texts, embs).commit()

r_res = ailake.search(res_path, embs[0].tolist(), top_k=5).to_pandas()
print('Residual PQ top-5:')
r_res[['row_id', 'distance']]


## 14. Deferred write — `write_batch_auto_deferred`

`write_batch_auto_deferred` returns immediately after persisting Parquet (~200k vec/s).
The index (HNSW or IVF-PQ, auto-selected) builds in a background thread.
Shard is served via flat scan until the index is ready — no downtime.

| Path | Throughput | Index ready |
|---|---|---|
| `write_batch` (inline) | ~6–10k vec/s | Immediately |
| `write_batch_auto_deferred` | ~200k vec/s | Background (~seconds) |


In [ ]:
import time
from ailake import TableWriter

deferred_path = str(BASE / 'nb_deferred')
t0 = time.perf_counter()

w = TableWriter(deferred_path, dim=DIM, metric='cosine')
w.write_batch_auto_deferred(texts, embs.tolist())
snap = w.commit()

elapsed = time.perf_counter() - t0
print(f'write_batch_auto_deferred: {elapsed*1000:.1f} ms for {len(texts)} rows (Parquet only, index in bg)')
print(f'Snapshot ID: {snap}')

# Search available immediately via flat scan fallback
r = ailake.search(deferred_path, embs[0].tolist(), top_k=3).to_pandas()
print()
print('Immediate search (flat scan while index builds):')
r[['row_id', 'distance']]


## 15. HNSW tuning — M and ef_construction

`hnsw_m` and `hnsw_ef_construction` control the HNSW graph quality per table.
No code change needed — stored in Iceberg table properties.

| Preset | `hnsw_m` | `hnsw_ef_construction` | Use case |
|---|---|---|---|
| Low latency | 8 | 100 | High QPS, recall@10 ~93 % |
| General (default) | 16 | 150 | Balanced |
| High recall | 24 | 200 | RAG, precision-critical |
| Max recall | 32 | 400 | Medical / legal |

`ef_search` at **query time** controls candidate pool size during graph traversal:
```python
ailake.search(path, query, top_k=10, ef_search=200)  # default: top_k * 5
```
Higher `ef_search` → better recall at cost of latency. `pruning_threshold` (0.0–1.0) discards geometrically distant files before any HNSW I/O — lower value = more aggressive pruning.


In [ ]:
# High-recall table (M=24, ef=200)
hr_path = str(BASE / 'nb_high_recall')
t = ailake.open_table(hr_path, dim=DIM, metric='normalized_cosine',
                      pre_normalize=True, hnsw_m=24, hnsw_ef_construction=200)
t.insert(texts, embs)
t.commit()
t   # HTML card shows hnsw_m and hnsw_ef_construction


In [ ]:
# Verify search works and shows distance ~0 for self-query
r = ailake.search(hr_path, embs[0].tolist(), top_k=1).to_list()[0]
print(f'Self-query distance: {r["distance"]:.2e}  (expected ≈ 0)')


# ef_search controls candidate pool at query time (default = top_k × 5)
r_hq = ailake.search(hr_path, embs[0].tolist(), top_k=3, ef_search=400).to_list()
print(f'ef_search=400 top result distance: {r_hq[0]["distance"]:.4f}')

# pruning_threshold cuts files whose centroid is geometrically far from query
# lower = more aggressive pruning; 1.0 = no pruning (default)
r_pruned = ailake.search(hr_path, embs[0].tolist(), top_k=3, pruning_threshold=0.7).to_list()
print(f'pruning_threshold=0.7 results: {len(r_pruned)} (may be fewer for small tables)')

## 16. Async API patterns

All search and write operations have async variants via `asyncio`.
Useful for FastAPI / async web services and concurrent multi-table search.


In [ ]:
import asyncio

async def multi_table_search(query, tables, top_k=3):
    """Search multiple tables concurrently — useful for sharded RAG."""
    tasks = [
        ailake.search(tbl, query, top_k=top_k).to_arrow_async()
        for tbl in tables
    ]
    results = await asyncio.gather(*tasks)
    return results

tables = [hnsw_path, pq_path, res_path]
q      = embs[0].tolist()

batches = await multi_table_search(q, tables, top_k=3)
for tbl, batch in zip(tables, batches):
    name = pathlib.Path(tbl).name
    print(f'{name:20s}  rows={batch.num_rows}  best_dist={batch.column("distance")[0].as_py():.4f}')


In [ ]:
# Async write via Table handle
async def async_write_demo():
    ap = str(BASE / 'nb_async_write')
    tbl = ailake.open_table(ap, dim=DIM, metric='cosine')
    await tbl.insert_async(texts[:10], embs[:10])
    snap = await tbl.commit_async()
    print(f'Async write committed: snapshot={snap}')

    rows = await ailake.search(ap, embs[0].tolist(), top_k=3).to_list_async()
    print(f'Async search: {len(rows)} results')
    return rows

rows = await async_write_demo()


## 17. Storage estimator — pure-math sizing

`ailake.estimate(rows, dim, hnsw_m=16, pq_m=None)` computes storage before writing
— no I/O, no actual data needed. Native binding (Fase 15) — same math the CLI's
`ailake estimate` subcommand runs. Useful for capacity planning before ingesting
large datasets.


In [ ]:
def print_estimate(rows: int, dim: int, hnsw_m: int = 16, pq_m: int | None = None) -> None:
    rows_out = ailake.estimate(rows, dim, hnsw_m=hnsw_m, pq_m=pq_m)

    def fmt(b):
        if b >= 1e9: return f'{b/1e9:.2f} GB'
        if b >= 1e6: return f'{b/1e6:.2f} MB'
        return f'{b/1024:.1f} KB'

    print(f'Storage estimate — {rows:,} rows x dim={dim} (native ailake.estimate())')
    for r in rows_out:
        note = f" — {r['note']}" if r['note'] else ''
        print(f"  {r['mode']:<24} total={fmt(r['total_bytes']):>10}  "
              f"({r['reduction_vs_f32_hnsw']:.1f}x smaller than F32+HNSW, recall {r['recall']}){note}")

# Demo fixture (dim=32)
print_estimate(rows=500, dim=32)
print()
# Real-world scale (dim=1536, text-embedding-3-small)
print_estimate(rows=1_000_000, dim=1536, hnsw_m=16, pq_m=48)


## 17B. Vector Precision — `precision=` (F16 / F32 / I8)

`TableWriter`/`open_table` accept `precision="f16"` (default) | `"f32"` | `"i8"` —
storage precision for the primary vector column (§6B of `CLAUDE.md`). Same three
modes the CLI's `ailake insert --precision` and `VectorColSpec.precision` (multi-column
writes, §21) already exposed — this section exercises the single-column `TableWriter`
path directly.

| Precision | Bytes/dim | Use case |
|---|---|---|
| `f32` | 4 | Max precision, no quantization loss |
| `f16` (default) | 2 | Recall loss <0.1% — recommended default |
| `i8` | 1 | Recall loss ~1-3%, smallest HNSW-compatible mode |


In [ ]:
import pathlib

prec_texts = [f'Precision demo doc {i}.' for i in range(200)]
prec_embs  = rng.standard_normal((200, DIM)).astype(np.float32)
prec_embs /= np.linalg.norm(prec_embs, axis=1, keepdims=True)

def dir_bytes(p):
    return sum(f.stat().st_size for f in pathlib.Path(p).rglob('*') if f.is_file())

prec_sizes = {}
prec_paths = {}
for precision in ('f32', 'f16', 'i8'):
    p = str(BASE / f'nb_precision_{precision}')
    ailake.open_table(p, dim=DIM, metric='cosine', precision=precision).insert(prec_texts, prec_embs).commit()
    prec_paths[precision] = p
    prec_sizes[precision] = dir_bytes(p)

print(f'{"precision":<10}{"total bytes":>14}  {"vs f16":>8}')
for precision in ('f32', 'f16', 'i8'):
    ratio = prec_sizes[precision] / prec_sizes['f16']
    print(f'{precision:<10}{prec_sizes[precision]:>14,}  {ratio:>7.2f}x')

print()
# Self-query sanity check — every precision mode must still find its own vector as nearest neighbour.
q = prec_embs[0].tolist()
for precision, p in prec_paths.items():
    r = ailake.search(p, q, top_k=1).to_list()[0]
    print(f'{precision}: top1 row_id={r["row_id"]}  distance={r["distance"]:.6f}')


## 18. Embedding Model Tracking

`embedding_model` and `embedding_model_version` record which model produced the
vectors — stored as `ailake.embedding-model` in Iceberg table properties and
per-file in the Avro manifest `key_metadata`.

Benefits:
- `ailake.search()` names the expected model in `ModelMismatch` errors when the query
  dimension doesn't match the table dimension.
- `migrate_embeddings()` reads per-file metadata to know which files need re-embedding
  when a table contains a mix of models.


In [ ]:
import json
import math
import random

MODEL_PATH = str(pathlib.Path(TABLE_PATH).parent / 'ailake_model_tracked')

# Write with model metadata attached
t = ailake.open_table(
    MODEL_PATH,
    dim=DIM,
    metric='cosine',
    embedding_model='synthetic-embed-v1',
    embedding_model_version='1.0',
)
t.insert(texts[:20], embs[:20])
snap = t.commit()
print(f'Snapshot: {snap}')


In [ ]:
# Inspect Iceberg metadata — ailake.embedding-model property
meta_dir  = pathlib.Path(MODEL_PATH) / 'default' / 'table' / 'metadata'
hint      = (meta_dir / 'version-hint.text').read_text().strip()
meta      = json.loads((meta_dir / f'v{hint}.metadata.json').read_text())
props     = meta.get('properties', {})

print('AI-Lake embedding model properties:')
for k in sorted(props):
    if k.startswith('ailake.embedding'):
        print(f'  {k:45s} = {props[k]}')


In [ ]:
# ModelMismatch error — triggered when query dim != table dim
wrong_query = np.random.default_rng(99).standard_normal(64).astype(np.float32)
try:
    ailake.search(MODEL_PATH, wrong_query, top_k=3).to_list()
except Exception as e:
    print(f'ModelMismatch caught: {e}')
    print('(includes expected model name from ailake.embedding-model property)')


## 19. Pattern B — auto-embed via `embed_fn`

Pass `embed_fn` to `open_table()` (or `TableWriter`) to avoid passing embeddings
on every `insert()` call. The SDK calls `embed_fn(texts)` automatically.

Useful for:
- Pipelines where embedding is a black-box service
- Prototyping — swap the model without changing the write loop
- Ensuring the tracked model always matches the embeddings that were written


In [ ]:
def synthetic_embed(texts: list[str]) -> list[list[float]]:
    """Deterministic synthetic embedding (dim=DIM) — replace with real model."""
    out = []
    for t in texts:
        rng = random.Random(hash(t) & 0xFFFFFFFF)
        v   = [rng.gauss(0, 1) for _ in range(DIM)]
        nm  = math.sqrt(sum(x * x for x in v)) or 1.0
        out.append([x / nm for x in v])
    return out

EMBED_FN_PATH = str(pathlib.Path(TABLE_PATH).parent / 'ailake_embed_fn')

# Pattern B: pass embed_fn — insert(texts) called WITHOUT embeddings
t = ailake.open_table(
    EMBED_FN_PATH,
    dim=DIM,
    metric='cosine',
    embed_fn=synthetic_embed,
    embedding_model='synthetic-embed-v1',
    embedding_model_version='1.0',
)
t.insert(texts[:10])   # no embeddings — SDK calls synthetic_embed automatically
snap = t.commit()
print(f'Pattern B snapshot: {snap}')

# Search works normally
q = synthetic_embed([texts[0]])[0]
top = ailake.search(EMBED_FN_PATH, q, top_k=1).to_list()[0]
print(f'Self-search distance: {top["distance"]:.2e}  (expected ≈ 0)')


## 20. Migration — `migrate_embeddings()`

`ailake.migrate_embeddings()` re-embeds all chunks in a table using a new model,
committing the result as a new Iceberg snapshot.

| Strategy | Peak storage | Downtime |
|---|---|---|
| `atomic_replace` | 1× (replaces file-by-file) | Brief mixed-model window |
| `dual_write_then_cutover` | 2× (writes new then swaps all at once) | **Zero** |

> `on_progress` receives keyword args: `files_done`, `files_total`, `rows_migrated`.


In [ ]:
def synthetic_embed_v2(texts: list[str]) -> list[list[float]]:
    """Simulated v2 model — different random seed, same dim."""
    out = []
    for t in texts:
        rng = random.Random((hash(t) ^ 0xDEADBEEF) & 0xFFFFFFFF)
        v   = [rng.gauss(0, 1) for _ in range(DIM)]
        nm  = math.sqrt(sum(x * x for x in v)) or 1.0
        out.append([x / nm for x in v])
    return out

progress_log = []

ailake.migrate_embeddings(
    path            = EMBED_FN_PATH,
    old_column      = 'embedding',
    new_column      = 'embedding',       # in-place model upgrade
    embed_fn        = synthetic_embed_v2,
    text_column     = 'text',
    strategy        = 'dual_write_then_cutover',
    new_model       = 'synthetic-embed-v2',
    new_model_version = '2.0',
    on_progress     = lambda *, files_done, files_total, rows_migrated: (
        progress_log.append({'files_done': files_done, 'files_total': files_total,
                             'rows_migrated': rows_migrated})
    ),
)

print(f'Migration done — {len(progress_log)} progress events')
for p in progress_log:
    print(f'  {p["files_done"]}/{p["files_total"]} files  {p["rows_migrated"]} rows')


In [ ]:
# Verify: ailake.embedding-model now shows v2 in Iceberg metadata
meta_dir  = pathlib.Path(EMBED_FN_PATH) / 'default' / 'table' / 'metadata'
hint      = (meta_dir / 'version-hint.text').read_text().strip()
meta      = json.loads((meta_dir / f'v{hint}.metadata.json').read_text())
props     = meta.get('properties', {})
print('Post-migration AI-Lake properties:')
for k in sorted(props):
    if k.startswith('ailake.embedding'):
        print(f'  {k:45s} = {props[k]}')


## 21. N vector columns — `VectorColSpec` + `write_batch_multi`

`write_batch_multi` writes a row set with **N independent vector columns** in one call.
Each column gets its own HNSW index in the AI-Lake file footer.

Use cases:
- Text + image embeddings from different models in the same row
- Dual text embeddings (`embedding` for content, `context_embedding` for positional context)
- Any multimodal scenario where the same document has heterogeneous representations

`VectorColSpec` declares each column: name, dim, metric, and optional modality tag.
Modality tags (`text` / `image` / `audio` / `video`) are stored as
`ailake.modality-<col>` in Iceberg table properties.


In [ ]:
import os, pathlib, json
import numpy as np
import ailake
from ailake import TableWriter, VectorColSpec

BASE          = pathlib.Path(os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')).parent
MM_PATH       = str(BASE / 'nb_multimodal')
TEXT_DIM      = 32
IMAGE_DIM     = 16   # synthetic CLIP-like image embeddings (smaller dim)
N             = 40
rng           = np.random.default_rng(7)

texts       = [f'Multimodal document {i}: a photo of a {["cat","dog","car","tree","sky"][i%5]}.' for i in range(N)]
text_embs   = rng.standard_normal((N, TEXT_DIM)).astype(np.float32)
text_embs  /= np.linalg.norm(text_embs, axis=1, keepdims=True)
image_embs  = rng.standard_normal((N, IMAGE_DIM)).astype(np.float32)
image_embs /= np.linalg.norm(image_embs, axis=1, keepdims=True)

# Declare two vector columns with modality tags
text_spec  = VectorColSpec('embedding',       TEXT_DIM,  'cosine', 'text')
image_spec = VectorColSpec('image_embedding', IMAGE_DIM, 'cosine', 'image')

# Write both columns in one call
w = TableWriter(MM_PATH, dim=TEXT_DIM, metric='cosine')
w.write_batch_multi(
    texts,
    [(text_spec, text_embs.tolist()), (image_spec, image_embs.tolist())],
)
snap = w.commit()
print(f'Multimodal table written: snapshot={snap}')
print(f'  Columns: {text_spec.column} (dim={TEXT_DIM}, modality=text)')
print(f'           {image_spec.column} (dim={IMAGE_DIM}, modality=image)')


In [ ]:
# Inspect Iceberg metadata — verify modality + per-column dim stored
meta_dir = pathlib.Path(MM_PATH) / 'default' / 'table' / 'metadata'
hint     = (meta_dir / 'version-hint.text').read_text().strip()
meta     = json.loads((meta_dir / f'v{hint}.metadata.json').read_text())
props    = meta.get('properties', {})

print('AI-Lake multimodal table properties:')
for k in sorted(props):
    if k.startswith('ailake.'):
        print(f'  {k:45s} = {props[k]}')


## 22. Cross-modal search — `search_multimodal` + RRF

`search_multimodal` runs an independent HNSW search per column, then fuses
ranked lists via **Reciprocal Rank Fusion**:

```
rrf_score = Σ  weight_i / (60 + rank_i)
```

Results ordered by descending `rrf_score` (higher = better match across all modalities).

| Query tuple | Meaning |
|---|---|
| `('embedding', text_vec, 0.7)` | Weight text similarity at 70 % |
| `('image_embedding', image_vec, 0.3)` | Weight image similarity at 30 % |

Weights need not sum to 1.0 — they're relative importance multipliers.


In [ ]:
# Pure text search — use text column only (weight=1.0)
text_query  = text_embs[0].tolist()
results_txt = ailake.search(MM_PATH, text_query, top_k=5).to_pandas()
print('Text-only search (single column):')
print(results_txt[['row_id', 'distance']])


In [ ]:
# Cross-modal RRF — text query 70 %, image query 30 %
image_query = image_embs[0].tolist()

results_rrf = ailake.search_multimodal(
    MM_PATH,
    queries=[
        ('embedding',       text_query,  0.7),
        ('image_embedding', image_query, 0.3),
    ],
    top_k=5,
)
import pandas as pd
df_rrf = pd.DataFrame(results_rrf)
print('Cross-modal RRF (text 70% + image 30%):')
df_rrf[['row_id', 'rrf_score']]


In [ ]:
# Weight ablation — compare 100/0 vs 70/30 vs 50/50 vs 0/100
def rrf_top5(tw, iw):
    r = ailake.search_multimodal(
        MM_PATH,
        queries=[('embedding', text_query, tw), ('image_embedding', image_query, iw)],
        top_k=5,
    )
    return [x['row_id'] for x in r]

configs = [(1.0, 0.0), (0.7, 0.3), (0.5, 0.5), (0.3, 0.7), (0.0, 1.0)]
print('Top-5 row_ids by weight (text_w / image_w):')
for tw, iw in configs:
    ids = rrf_top5(tw, iw)
    print(f'  {tw:.1f}/{iw:.1f} → {ids}')


## 23. `MultimodalContextSchema` — canonical column names

`ailake.multimodal_columns` (Rust) / `ailake.MULTIMODAL_COLUMNS` (Python) provides
canonical string constants for media table columns. Use them instead of raw
string literals to avoid typos and make schemas self-documenting.

| Constant | Value | Purpose |
|---|---|---|
| `MEDIA_URI` | `"media_uri"` | S3/GCS/HTTPS URI of the raw asset |
| `MEDIA_MIME` | `"media_mime"` | MIME type (e.g. `"image/jpeg"`) |
| `MEDIA_CAPTION` | `"media_caption"` | Caption from a vision/audio model |
| `IMAGE_EMBEDDING` | `"image_embedding"` | CLIP/SigLIP image column |
| `AUDIO_TRANSCRIPT` | `"audio_transcript"` | Whisper transcript |
| `THUMBNAIL_B64` | `"thumbnail_b64"` | Base64 JPEG thumbnail (≤ 64×64) |

> AI-Lake is **not a blob store** — media files live in object storage.
> Only URIs and embeddings belong in an AI-Lake table.


In [ ]:
# Access column name constants via ailake.MULTIMODAL_COLUMNS dict
import ailake

MCOLS = {
    'MEDIA_URI':        'media_uri',
    'MEDIA_MIME':       'media_mime',
    'MEDIA_CAPTION':    'media_caption',
    'IMAGE_EMBEDDING':  'image_embedding',
    'AUDIO_TRANSCRIPT': 'audio_transcript',
    'THUMBNAIL_B64':    'thumbnail_b64',
}

print('Multimodal column name constants:')
for const, val in MCOLS.items():
    print(f'  MULTIMODAL_COLUMNS.{const:20s} = "{val}"')

# Typical multimodal row schema (Arrow)
import pyarrow as pa
schema = pa.schema([
    pa.field('chunk_id',          pa.string()),
    pa.field('chunk_text',        pa.string()),
    pa.field('embedding',         pa.binary()),     # text F16, dim=1536
    pa.field('image_embedding',   pa.binary()),     # image F16, dim=512
    pa.field('media_uri',         pa.string()),     # s3://bucket/key.jpg
    pa.field('media_mime',        pa.string()),     # image/jpeg
    pa.field('media_caption',     pa.string()),     # BLIP-2 caption
    pa.field('audio_transcript',  pa.string()),     # Whisper transcript (null for images)
    pa.field('thumbnail_b64',     pa.string()),     # base64 JPEG 64×64
])
print()
print('Example MultimodalContextSchema Arrow schema:')
for field in schema:
    print(f'  {field.name:22s}: {field.type}')


## 24. Agent memory — `ailake.Agent`

`ailake.Agent` is a high-level helper for AI agent frameworks (LangChain, CrewAI, AutoGen).
It wraps `TableWriter` + `search` + `assemble_context` behind four simple methods:

| Method | Description |
|---|---|
| `remember(text, importance=1.0)` | Embed and persist a memory chunk |
| `recall(query, top_k=5)` | Vector search over this agent's memories |
| `log_tool_call(name, input, output, outcome, latency_ms)` | Store a tool call record |
| `assemble_context(query, max_tokens)` | RAG context XML for this agent's partition |

Each agent is isolated by `agent_id` using Iceberg identity partitioning — searches
never cross agent boundaries without an explicit `partition_filter=None`.

> The `init_demo.py` fixture pre-wrote 50 rows each for `agent-A` and `agent-B`
> to `/data/ailake_agent`. This section uses that fixture.

In [ ]:
import math, random, os, pathlib, json
import numpy as np
import ailake

AGENT_PATH = os.environ.get('DEMO_AGENT_PATH', '/data/ailake_agent')
DIM_AGENT  = int(json.load(open(pathlib.Path(AGENT_PATH).parent / 'demo_query.json'))['dim'])

def demo_embed(texts):
    """Deterministic synthetic embed — replace with real model in production."""
    out = []
    for t in texts:
        rng = random.Random(hash(t) & 0xFFFFFFFF)
        v   = [rng.gauss(0, 1) for _ in range(DIM_AGENT)]
        nm  = math.sqrt(sum(x * x for x in v)) or 1.0
        out.append([x / nm for x in v])
    return out

# Create agent helper — backed by the pre-built fixture partition
agent_a = ailake.Agent(AGENT_PATH, embed_fn=demo_embed, agent_id='agent-A')
print(f'Agent created: {agent_a}')

# remember() embeds + commits a new memory chunk
snap = agent_a.remember('Distributed vector search with HNSW requires careful shard sizing.', importance=0.9)
print(f'  remember() → snapshot_id={snap}')

# recall() searches only within agent-A's partition
results = agent_a.recall('HNSW shard sizing', top_k=3)
print(f'\nagent-A recall (top-3):')
for r in results:
    preview = r.get('text', r.get('chunk_text', ''))[:70]
    print(f'  dist={r["distance"]:.4f}  text="{preview}..."')

## 25. Partition isolation — `partition_by` / `partition_filter`

Writing with `partition_by="agent_id"` creates an Iceberg identity partition.
Each `partition_value` writes to its own file set — Iceberg manifest pruning
eliminates other partitions before centroid geometry is even checked.

Searching with `partition_filter="agent-A"` restricts results to files where
`agent_id == "agent-A"`. No rows from `agent-B` are ever loaded.

| API | Default | Effect |
|---|---|---|
| `TableWriter(partition_by=col, partition_value=val)` | None | Writes to isolated shard |
| `ailake.search(..., partition_filter=val)` | None (all partitions) | Prune at manifest level |
| `Agent.recall()` | agent_id auto-set | Always partition-scoped |

In [ ]:
with open(pathlib.Path(AGENT_PATH).parent / 'demo_query.json') as f:
    demo_meta = json.load(f)

query_vec = np.array(demo_meta['query_vector'], dtype=np.float32)

# Search WITHOUT partition_filter — returns rows from both agents
all_results = ailake.search(AGENT_PATH, query_vec, top_k=10).to_pandas()
print(f'No filter  — {len(all_results)} rows returned (both partitions)')

# Search WITH partition_filter — only agent-A rows
a_results = ailake.search(AGENT_PATH, query_vec, top_k=10,
                           partition_filter='agent-A').to_pandas()
print(f'agent-A    — {len(a_results)} rows returned')

# Search WITH partition_filter — only agent-B rows
b_results = ailake.search(AGENT_PATH, query_vec, top_k=10,
                           partition_filter='agent-B').to_pandas()
print(f'agent-B    — {len(b_results)} rows returned')

# Partition isolation: files from agent-A and agent-B partitions never overlap.
# NOTE: row_id is a per-file sequential index (starts at 0 in every shard) so
#       row_id overlap is expected and does NOT indicate a partition leak.
#       The correct isolation check is on the 'file' column.
a_files = set(a_results['file'].tolist())
b_files = set(b_results['file'].tolist())
file_overlap = a_files & b_files
print(f'\nFile overlap between partitions: {len(file_overlap)} (expected 0)')
assert len(file_overlap) == 0, f'Partition leak detected! Shared files: {file_overlap}'
print('PASS — partitions are fully isolated')

## 26. Hybrid scoring — `ScoreFn`

`SearchConfig.score_fn` injects a custom scoring function into the merge step.
Called once per candidate result after HNSW search:

```python
ScoreFn = Callable[[distance: float, row: dict], float]
```

Typical hybrid scores:

| Formula | Use case |
|---|---|
| `distance` | Pure vector similarity (default) |
| `distance * recency_weight` | Recency-weighted retrieval |
| `distance * importance_score` | Importance-weighted retrieval |
| `distance * recency_weight * importance_score` | Full episodic hybrid |

Lower `score_fn` output = ranked higher (matches HNSW distance convention).

In [ ]:
import time, shutil

# Write a small table with recency_weight and importance_score columns (EpisodicMemorySchema fields)
HYBRID_PATH = str(pathlib.Path(AGENT_PATH).parent / 'nb_hybrid_scoring')
from ailake import TableWriter

# Clean up any stale data from previous runs to avoid schema conflicts
shutil.rmtree(HYBRID_PATH, ignore_errors=True)

n = 20
texts_h  = [f'Memory {i}: discussion about topic {i % 5}.' for i in range(n)]
embs_h   = np.random.default_rng(55).standard_normal((n, DIM_AGENT)).astype(np.float32)
embs_h  /= np.linalg.norm(embs_h, axis=1, keepdims=True)

# Simulate recency_weight (1.0 = fresh, 0.1 = stale) and importance_score (0-1)
import math as _math
recency_weights   = [_math.exp(-0.05 * i) for i in range(n)]  # decays with age
importance_scores = [0.9 if i % 5 == 0 else 0.3 for i in range(n)]

w = TableWriter(HYBRID_PATH, dim=DIM_AGENT, metric='cosine')
w.write_batch(
    texts_h,
    embs_h.tolist(),
    extra_columns={
        'mem_idx':          list(range(n)),   # explicit index (row_id is system-internal)
        'recency_weight':   recency_weights,
        'importance_score': importance_scores,
    },
)
w.commit()
print(f'Hybrid table written: {n} rows with recency_weight + importance_score')

# ScoreFn: combine distance with recency × importance
def hybrid_score(distance: float, row: dict) -> float:
    rw = row.get('recency_weight',   1.0)
    im = row.get('importance_score', 1.0)
    return distance / max(rw * im, 1e-6)

# Compare pure-distance vs hybrid ranking
pure_top5   = ailake.search(HYBRID_PATH, embs_h[0], top_k=5, fetch_data=True).to_pandas()
hybrid_top5 = ailake.search(HYBRID_PATH, embs_h[0], top_k=5, fetch_data=True,
                             score_fn=hybrid_score).to_pandas()

import pandas as pd
comparison = pd.DataFrame({
    'pure_mem_idx'  : pure_top5['mem_idx'].values,
    'hybrid_mem_idx': hybrid_top5['mem_idx'].values,
})
print('\nPure vs hybrid top-5 mem_idx:')
print(comparison)

## 27. `ToolCallSchema` — agent tool call logging

`ToolCallSchema` extends `LlmContextSchema` with agent execution fields.
Log every tool invocation so you can later ask: *"when did the agent use web_search
on a query similar to this one, and what did it return?"*

| Column | Type | Description |
|---|---|---|
| `agent_id` | string | Agent identifier |
| `session_id` | string | Conversation / run session |
| `step_index` | uint32 | Step number within the session |
| `tool_name` | string | Name of the tool called |
| `tool_input_json` | string | JSON-serialised input |
| `tool_output_json` | string | JSON-serialised output |
| `outcome` | string | `"success"` / `"failure"` / `"timeout"` |
| `latency_ms` | uint32 | Wall-clock latency |
| `chunk_text` | string | Concatenated input+output for embedding |
| `embedding` | Vector | Semantic vector of the call context |

Store via `agent.log_tool_call()` or `TableWriter` with the full column set.
Recall via `ailake.search()` with `partition_filter=agent_id`.

In [ ]:
TOOL_PATH = str(pathlib.Path(AGENT_PATH).parent / 'nb_tool_calls')

# Simulate a session with 5 tool calls
tool_calls = [
    ('web_search',  {'q': 'HNSW algorithm'},      {'snippets': ['HNSW is a graph-based ANN method.']}, 'success', 230),
    ('web_search',  {'q': 'IVF-PQ quantization'},  {'snippets': ['IVF-PQ uses product quantization.']}, 'success', 180),
    ('code_exec',   {'code': 'print(42)'},          {'stdout': '42'},                                   'success',  12),
    ('web_search',  {'q': 'pgvector performance'},  {'snippets': ['pgvector is limited to ~1M rows.']}, 'success', 310),
    ('code_exec',   {'code': 'import ailake'},       {'error': 'ModuleNotFoundError'},                  'failure',   5),
]

# Build chunk_text = "tool_name: input | output" for embedding
tc_texts = [
    f'{name}: {json.dumps(inp)} | {json.dumps(out)}'
    for name, inp, out, outcome, latency in tool_calls
]
tc_embs = np.array(demo_embed(tc_texts), dtype=np.float32)

w = TableWriter(TOOL_PATH, dim=DIM_AGENT, metric='cosine',
                partition_by='agent_id', partition_value='agent-A')
w.write_batch(
    tc_texts, tc_embs.tolist(),
    extra_columns={
        'agent_id'       : ['agent-A'] * len(tool_calls),
        'session_id'     : ['session-001'] * len(tool_calls),
        'step_index'     : list(range(len(tool_calls))),
        'tool_name'      : [c[0] for c in tool_calls],
        'tool_input_json': [json.dumps(c[1]) for c in tool_calls],
        'tool_output_json': [json.dumps(c[2]) for c in tool_calls],
        'outcome'        : [c[3] for c in tool_calls],
        'latency_ms'     : [c[4] for c in tool_calls],
    },
)
w.commit()
print(f'Logged {len(tool_calls)} tool calls to {TOOL_PATH}')

# Semantic search: "find past tool calls related to vector search"
query_emb = demo_embed(['vector search ANN algorithms'])[0]
results   = ailake.search(TOOL_PATH, query_emb, top_k=3,
                           partition_filter='agent-A', fetch_data=True).to_pandas()
print('\nTool calls most similar to "vector search ANN algorithms":')
for _, row in results.iterrows():
    print(f'  [{row["tool_name"]}] {row["text"][:80]}...')

## 28. `EpisodicMemorySchema` — recency decay scoring

`EpisodicMemorySchema` adds three columns on top of `LlmContextSchema`:

| Column | Type | Description |
|---|---|---|
| `recency_weight` | float32 | `exp(-λ × days_since_access)` — decays toward 0 with age |
| `access_count` | uint32 | How many times this memory was recalled |
| `importance_score` | float32 | Agent-assigned relevance signal (0–1) |
| `last_accessed_at` | timestamp | Updated by `MemoryDecayJob` |

The hybrid score for ranking:

```
final_score = distance × recency_weight × importance_score
```

Lower `final_score` = ranked first (same convention as plain distance).
`MemoryDecayJob` (Phase 9, pending) recomputes `recency_weight` periodically
and persists the update via Iceberg compaction.

In [ ]:
import math as _math, datetime, shutil

EPISODIC_PATH = str(pathlib.Path(AGENT_PATH).parent / 'nb_episodic')

# Clean up stale data from previous runs to avoid schema conflicts
shutil.rmtree(EPISODIC_PATH, ignore_errors=True)

# Simulate 10 memories with different ages and importance levels
now   = datetime.datetime.now(datetime.UTC)
decay_lambda = 0.1  # recency_weight = exp(-λ × days_old)

n_ep       = 10
ep_texts   = [f'Episodic memory {i}: observations about pattern {i % 3}.' for i in range(n_ep)]
ep_embs    = np.random.default_rng(77).standard_normal((n_ep, DIM_AGENT)).astype(np.float32)
ep_embs   /= np.linalg.norm(ep_embs, axis=1, keepdims=True)

days_old         = [0, 0, 1, 2, 5, 10, 15, 20, 30, 60]   # age of each memory
recency_weights  = [_math.exp(-decay_lambda * d) for d in days_old]
access_counts    = [5, 0, 3, 1, 0, 2, 0, 1, 0, 0]
importance_scores = [0.9 if i % 3 == 0 else 0.4 for i in range(n_ep)]

w = TableWriter(EPISODIC_PATH, dim=DIM_AGENT, metric='cosine',
                partition_by='agent_id', partition_value='agent-A')
w.write_batch(
    ep_texts, ep_embs.tolist(),
    extra_columns={
        'mem_idx'         : list(range(n_ep)),  # explicit index (row_id is system-internal)
        'agent_id'        : ['agent-A'] * n_ep,
        'recency_weight'  : recency_weights,
        'access_count'    : access_counts,
        'importance_score': importance_scores,
        'last_accessed_at': [(now - datetime.timedelta(days=d)).isoformat() for d in days_old],
    },
)
w.commit()
print('Episodic memories written:')
for i, (d, rw, im) in enumerate(zip(days_old, recency_weights, importance_scores)):
    print(f'  [{i}] days_old={d:2d}  recency_weight={rw:.3f}  importance={im:.1f}')

# Hybrid ScoreFn — favour fresh + important memories
def episodic_score(distance: float, row: dict) -> float:
    rw = row.get('recency_weight',   1.0)
    im = row.get('importance_score', 1.0)
    return distance / max(rw * im, 1e-9)

# Query — compare pure distance vs episodic hybrid ranking
q_ep = ep_embs[5].tolist()   # row 5 is "stale" (days_old=10)

pure     = ailake.search(EPISODIC_PATH, q_ep, top_k=5, fetch_data=True,
                          partition_filter='agent-A').to_pandas()
episodic = ailake.search(EPISODIC_PATH, q_ep, top_k=5, fetch_data=True,
                          partition_filter='agent-A', score_fn=episodic_score).to_pandas()

import pandas as pd
cmp = pd.DataFrame({
    'pure_mem_idx'    : pure['mem_idx'].values,
    'episodic_mem_idx': episodic['mem_idx'].values,
})
print('\nRanking comparison (pure distance vs episodic hybrid):')
print(cmp)
print('\nNote: stale memories (high days_old) are downranked in episodic mode.')

## 29. `delete_where` — Iceberg equality delete

`ailake.delete_where(table_path, column, values)` writes an Iceberg **equality delete file** and commits a Delete snapshot. No data files are rewritten — existing rows are masked at scan time (logical delete, ACID-safe).

Works with any indexed column. The demo table has 100 rows with rows 0-9 pre-deleted at container start.


In [ ]:
import ailake, ailake as _al, pathlib, os, json, shutil
import numpy as np

with open(QUERY_PATH) as f:
    demo = json.load(f)

# Recreate delete_demo fresh each run so the before/after counts are deterministic.
DEL_PATH = str(pathlib.Path(demo["table_paths"]["delete_demo"]).parent / "nb_delete_demo")
shutil.rmtree(DEL_PATH, ignore_errors=True)

DIM = int(demo["dim"])
rng = np.random.default_rng(42)
embs = rng.standard_normal((100, DIM)).astype(np.float32)
embs /= np.linalg.norm(embs, axis=1, keepdims=True)
texts = [f"Delete-demo doc {i}." for i in range(100)]

w = ailake.TableWriter(DEL_PATH, dim=DIM, metric=demo["metric"])
w.write_batch(texts, embs.tolist(), extra_columns={"id": [str(i) for i in range(100)]})
w.commit()
ailake.delete_where(DEL_PATH, "id", [str(i) for i in range(10)])  # pre-delete rows 0-9

q = demo["query_vector"]

# Count rows before additional delete (10 already pre-deleted)
before = ailake.search(DEL_PATH, q, top_k=100).to_list()
print(f"Visible rows before delete: {len(before)} (of 100 written, 10 pre-deleted, expected 90)")

# Delete rows 10-19 — should reduce visible results
ailake.delete_where(DEL_PATH, "id", [str(i) for i in range(10, 20)])
print("Deleted rows 10-19 via equality delete file.")

after = ailake.search(DEL_PATH, q, top_k=100).to_list()
print(f"Visible rows after delete:  {len(after)} (expected 80; rows 0-19 now masked)")
assert len(after) < len(before), f"delete did not reduce visible rows ({len(before)} -> {len(after)})"
assert len(before) == 90, f"expected 90 before delete, got {len(before)}"
assert len(after) == 80, f"expected 80 after delete, got {len(after)}"
print("PASS — equality delete works correctly.")

# Empty values list is always a no-op (no file written)
ailake.delete_where(DEL_PATH, "id", [])
print("Empty values list → no-op (no snapshot written).")


## 30. Schema evolution — `add_column` / `rename_column`

AI-Lake exposes Iceberg schema evolution as two functions:

- `ailake.add_column(path, name, iceberg_type, ...)` — adds an optional column, metadata-only
- `ailake.rename_column(path, old_name, new_name)` — renames by field ID (data files untouched)

Field IDs are stable — existing Parquet files continue to be readable after a rename.

`ailake.evolve_schema(path, add_columns=[...], rename_columns=[...])` is a combined wrapper that applies additions and renames in one call.


In [ ]:
import ailake

with open(QUERY_PATH) as f:
    demo = json.load(f)

EVO_PATH = demo["table_paths"]["schema_evo"]

# add_column — source_url was already added at init; add another one here
schema_id = ailake.add_column(
    EVO_PATH,
    name="quality_score",
    iceberg_type="float",
    required=False,
    initial_default=0.0,   # new files write 0.0 when not supplied
)
print(f"add_column('quality_score', float): new_schema_id={schema_id}")

# rename_column — metadata-only, no data rewritten
schema_id2 = ailake.rename_column(EVO_PATH, "quality_score", "relevance_score")
print(f"rename_column(quality_score → relevance_score): new_schema_id={schema_id2}")

# Verify: search still works after schema evolution
results = ailake.search(EVO_PATH, demo["query_vector"], top_k=3).to_list()
print(f"Search after evolution: {len(results)} results, top row_id={results[0]['row_id']}")


# evolve_schema() — combined add + rename in one call
schema_id3 = ailake.evolve_schema(
    EVO_PATH,
    add_columns=[{"name": "ingest_ts", "type": "long", "required": False}],
    rename_columns=[{"from": "relevance_score", "to": "score"}],
)
print(f"evolve_schema (add ingest_ts + rename): new_schema_id={schema_id3}")

## 30B. Backfilling a vector column — `add_vector_column` + `backfill_vector_column`

Adds a *second* vector column to an existing table's schema — metadata-only, no
data files rewritten — then backfills it into every existing file. Old files
return `null` for the new column until backfilled; the two-step split lets you
declare the schema change immediately and run the (expensive) rewrite as a
separate, resumable job. Useful for re-embedding into a new model without a
full `migrate_embeddings()` cutover, or for adding a second modality later.

Searching the new column requires `ailake.search_multimodal()` — plain
`ailake.search()` always targets the table's primary vector column.


In [ ]:
import pathlib

BACKFILL_PATH = str(BASE / 'nb_backfill_demo')

backfill_texts = [f'Backfill demo doc {i}.' for i in range(30)]
backfill_embs  = rng.standard_normal((30, DIM)).astype(np.float32)
backfill_embs /= np.linalg.norm(backfill_embs, axis=1, keepdims=True)

ailake.open_table(BACKFILL_PATH, dim=DIM, metric='cosine').insert(backfill_texts, backfill_embs).commit()

# 1. Declare the new column — metadata-only; existing files return null for it until backfilled
new_schema_id = ailake.add_vector_column(
    BACKFILL_PATH, column='embedding_v2', dim=DIM, metric='cosine', precision='f32',
)
print(f'add_vector_column(embedding_v2): new_schema_id={new_schema_id}')

# 2. Backfill — rewrites every file, computing embedding_v2 via embed_fn over text_column='text'
def synthetic_embed_v2(texts: list[str]) -> list[list[float]]:
    out = []
    for t in texts:
        vec = rng.standard_normal(DIM).astype(np.float32)
        vec /= np.linalg.norm(vec)
        out.append(vec.tolist())
    return out

ailake.backfill_vector_column(BACKFILL_PATH, 'embedding_v2', synthetic_embed_v2, text_column='text')
print('backfill_vector_column(embedding_v2): done — every file now carries both columns')

# Verify — search the new column via search_multimodal (plain search() only targets the primary column)
q2 = synthetic_embed_v2(['probe'])[0]
results = ailake.search_multimodal(BACKFILL_PATH, [('embedding_v2', q2, 1.0)], top_k=3)
print(f'search_multimodal on embedding_v2 after backfill: {len(results)} results')
for r in results:
    print(f"  row_id={r['row_id']:3d}  rrf_score={r['rrf_score']:.4f}")


## 31. Compaction — `ailake.compact()`

`ailake.compact()` merges small Parquet files and rebuilds HNSW/IVF-PQ indexes.
Run after many small inserts or after `delete_where` to reclaim space and maintain search performance.

```
Small files → merged file (new HNSW inside) → old entries tombstoned in Iceberg manifest
```

Compaction is idempotent and safe to run at any time.

In [ ]:
import ailake, pathlib, os, json

with open(QUERY_PATH) as f:
    demo = json.load(f)

COMPACT_PATH = demo["table_paths"]["hnsw"]  # use main table

# Compact small files — no-op if already optimal
# Note: compact() requires the ailake CLI binary; skip gracefully if not installed.
try:
    result = ailake.compact(COMPACT_PATH)
    print(f"compact() result: files_compacted={result.get('files_compacted', 0)}")
except (FileNotFoundError, OSError) as e:
    print(f"compact() skipped — CLI binary not available in this environment ({e})")
    print("Install the ailake CLI via: cargo install ailake-cli")

# Verify search still works after compaction
results = ailake.search(COMPACT_PATH, demo["query_vector"], top_k=3).to_list()
print(f"Post-compaction search: {len(results)} results, top distance={results[0]['distance']:.4f}")

## 31. Partition fields + `format_version=3`

`TableWriter(partition_fields=[...], format_version=3)` creates an Iceberg v3 table with explicit partition specs. Unlike `partition_by` (hidden single-column partition), `partition_fields` supports:

- Multiple columns
- Any Iceberg transform: `identity`, `year`, `month`, `day`, `hour`, `bucket[N]`, `truncate[N]`
- Full Iceberg partition spec stored in `metadata.json`

The demo fixture created a table partitioned by `topic_id` (identity, int) with format_version=3.


In [ ]:
import ailake, pathlib, json, os

with open(QUERY_PATH) as f:
    demo = json.load(f)

PV3_PATH = demo["table_paths"]["partitioned_v3"]
DIM = int(demo["dim"])

# Search works identically — partition pruning is transparent
q = demo["query_vector"]
results = ailake.search(PV3_PATH, q, top_k=5).to_list()
print(f"Partitioned v3 search: {len(results)} results")
for r in results:
    print(f"  row_id={r['row_id']:3d}  distance={r['distance']:.4f}")

# Inspect Iceberg metadata — format-version and partition spec
meta_dir = pathlib.Path(PV3_PATH) / "default" / "table" / "metadata"
hint = (meta_dir / "version-hint.text").read_text().strip()
meta = json.loads((meta_dir / f"v{hint}.metadata.json").read_text())
print(f"\nformat-version:   {meta['format-version']}")
print(f"partition-specs:  {meta.get('partition-specs', meta.get('partition-spec', 'n/a'))}")
print(f"ailake properties: { {k:v for k,v in meta.get('properties',{}).items() if k.startswith('ailake')} }")

# Create a brand-new partitioned table inline (format_version=3, bucket transform)
import numpy as np, tempfile
rng = np.random.default_rng(seed=77)
NEW_PATH = str(pathlib.Path(PV3_PATH).parent / "nb_partitioned_bucket")
w = ailake.TableWriter(
    NEW_PATH,
    dim=DIM,
    metric="cosine",
    partition_fields=[("shard_id", "bucket[4]", "int")],  # 4 buckets
    format_version=3,
)
txts = [f"doc_{i}" for i in range(20)]
embs = [rng.standard_normal(DIM).tolist() for _ in range(20)]
shard_ids = [i % 4 for i in range(20)]
w.write_batch(txts, embs, extra_columns={"shard_id": shard_ids})
snap_id = w.commit()
print(f"\nbucket[4] table: snapshot_id={snap_id}, {len(txts)} rows")


## 32. Tantivy FTS — `fts_text_columns` + `search_text()`

`fts_text_columns` opt-in writes a Tantivy inverted index per file (`AILK_FTS` section, zstd).
`search_text()` uses O(log N) Tantivy fast path when the index is present — falls back to
BM25 brute-force for legacy files.

> For the full FTS demo (multi-column, query syntax, FTS+HNSW hybrid re-ranking),
> open **`11_fts.ipynb`**.


In [ ]:
import os, pathlib, shutil
import numpy as np
import ailake

BASE    = pathlib.Path(os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')).parent
DIM     = 32
rng     = np.random.default_rng(99)
texts   = [f'FTS demo doc {i}: topic {["rust","async","vector","iceberg","search"][i%5]} content.' for i in range(20)]
embs    = rng.standard_normal((20, DIM)).astype(np.float32)
embs   /= np.linalg.norm(embs, axis=1, keepdims=True)

FTS_PATH = str(BASE / 'nb_fts_intro')
shutil.rmtree(FTS_PATH, ignore_errors=True)

# Write with Tantivy FTS enabled on 'text' column (TableWriter supports fts_text_columns)
w = ailake.TableWriter(FTS_PATH, dim=DIM, metric='cosine', fts_text_columns=['text'])
w.write_batch(texts, embs.tolist())
w.commit()
print('FTS table written.')

# Pure full-text search — no embedding needed
# search_text() returns dicts with 'distance' field (negated BM25/Tantivy score)
hits = ailake.search_text(FTS_PATH, 'rust async', top_k=5, text_column='text')
print(f'search_text("rust async") → {len(hits)} hits')
for h in hits:
    print(f'  row_id={h["row_id"]:3d}  score={-h["distance"]:.4f}')

## Next Steps

| Resource | Location |
|---|---|
| File format binary spec | `docs/specs/FILE_FORMAT.md` |
| Python API reference | `ailake-py/README.md` |
| Iceberg catalog integration | `docs/architecture/CATALOG_BACKENDS.md` |
| Cloud deployment (S3/GCS/Azure) | `docs/specs/CLOUD_DEPLOY.md` |
| JVM plugins (Spark/Trino/Flink) | `docs/specs/JVM_PLUGINS.md` |
| DuckDB deep-dive | `02_duckdb.ipynb` |
| Apache Spark | `03_spark.ipynb` |
| Trino (requires `--profile engines`) | `04_trino.ipynb` |
| BigQuery emulator (requires `--profile engines`) | `05_bigquery.ipynb` |
| Multimodal deep-dive | `07_multimodal.ipynb` |
| **Agent memory deep-dive (Phase 9)** | **`08_agents.ipynb`** |
| **Hybrid BM25+vector search** | **`09_hybrid_search.ipynb`** |
| **Tantivy FTS deep-dive (Phase T)** | **`11_fts.ipynb`** |
| **Apache Airflow pipelines** | **`12_airflow.ipynb`** |